# Speed Dating data
Can't know how to clean it


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.options.display.max_rows = 1000
pd.options.display.max_columns = 200
pd.set_option('display.max_columns', None)

In [ ]:
#%pip install ydata-profiling

In [ ]:
# Load data
data = pd.read_csv('Speed_Dating_Data.csv', sep=',', encoding='ISO-8859-1')
data.head(20)

In [ ]:
data.isnull().sum()

In [ ]:
data.isnull().mean() * 100

In [ ]:
# 1. BẢNG THÔNG TIN CÁ NHÂN (PERSON_INFO)
# Chứa các đặc điểm nhân khẩu học, học vấn và lối sống (Cố định cho mỗi iid)
# =================================================================
personal_cols = [
    'iid', 'gender', # Định danh và nhóm thử nghiệm
    'age', 'field', 'field_cd', 'undergra',         # Học vấn: Tuổi, ngành học, trường ĐH
    # 'mn_sat', 'tuition', 
    'race', 'imprace',          # Kinh tế/Sắc tộc: Điểm SAT, học phí, tầm quan trọng sắc tộc
    'imprelig', 'from', 'zipcode', 'income',        # Tôn giáo, quê quán, thu nhập gia đình
    'goal', 'date', 'go_out', 'career_c', # Mục tiêu tham gia, tần suất hẹn hò, nghề nghiệp
    'sports', 'tvsports', 'exercise', 'dining',     # Sở thích (1-10): Thể thao, ăn uống...
    'museums', 'art', 'hiking', 'gaming', 'clubbing', 
    'reading', 'tv', 'theater', 'movies', 'concerts', 
    'music', 'shopping', 'yoga',
    # 'exphappy', # Kỳ vọng mức độ hạnh phúc (1-10)
    # 'expnum',   # Dự đoán số người sẽ thích mình (trong 20 người)
    # 'match_es', # Dự đoán số lượng match sẽ có
    'attr1_1', 'sinc1_1', 'intel1_1', 'fun1_1', 'amb1_1', 'shar1_1', # Tìm kiếm ở đối phương

    'attr3_1', 'sinc3_1', 'intel3_1', 'fun3_1', 'amb3_1',                   # Tự đánh giá mình
]



In [95]:
interaction_cols = [
    'iid', 'pid', 'int_corr', 'samerace',
    # Thông tin đối tác (hậu tố _o)
    'dec_o', 'attr_o', 'sinc_o', 'intel_o', 'fun_o', 'amb_o', 'shar_o', 'like_o', 'prob_o',
    # Đánh giá của chính người đó về đối tác (Scorecard)
    'dec', 'attr', 'sinc', 'intel', 'fun', 'amb', 'shar', 'like', 'prob',
    'match' # Kết quả liên lạc thực tế
] 

In [ ]:
data[interaction_cols].drop_duplicates().head(20)

In [96]:
# Tập hợp tất cả các cột đã phân loại
classified_cols = set(personal_cols + interaction_cols)

# Các cột còn lại chưa được đưa vào bảng nào
remaining_cols = [col for col in data.columns if col not in classified_cols]

print(f"Các cột chưa phân loại: {remaining_cols}")

Các cột chưa phân loại: ['id', 'idg', 'condtn', 'wave', 'round', 'position', 'positin1', 'order', 'partner', 'age_o', 'race_o', 'pf_o_att', 'pf_o_sin', 'pf_o_int', 'pf_o_fun', 'pf_o_amb', 'pf_o_sha', 'met_o', 'mn_sat', 'tuition', 'career', 'exphappy', 'expnum', 'attr4_1', 'sinc4_1', 'intel4_1', 'fun4_1', 'amb4_1', 'shar4_1', 'attr2_1', 'sinc2_1', 'intel2_1', 'fun2_1', 'amb2_1', 'shar2_1', 'attr5_1', 'sinc5_1', 'intel5_1', 'fun5_1', 'amb5_1', 'met', 'match_es', 'attr1_s', 'sinc1_s', 'intel1_s', 'fun1_s', 'amb1_s', 'shar1_s', 'attr3_s', 'sinc3_s', 'intel3_s', 'fun3_s', 'amb3_s', 'satis_2', 'length', 'numdat_2', 'attr7_2', 'sinc7_2', 'intel7_2', 'fun7_2', 'amb7_2', 'shar7_2', 'attr1_2', 'sinc1_2', 'intel1_2', 'fun1_2', 'amb1_2', 'shar1_2', 'attr4_2', 'sinc4_2', 'intel4_2', 'fun4_2', 'amb4_2', 'shar4_2', 'attr2_2', 'sinc2_2', 'intel2_2', 'fun2_2', 'amb2_2', 'shar2_2', 'attr3_2', 'sinc3_2', 'intel3_2', 'fun3_2', 'amb3_2', 'attr5_2', 'sinc5_2', 'intel5_2', 'fun5_2', 'amb5_2', 'you_call', 'th

In [111]:
df = data.drop(columns=[col for col in data.columns if col not in classified_cols]).copy()
df = df.drop(df.index[df['pid'].isnull()])  # Loại bỏ các hàng có pid bị thiếu
df['income'] = pd.to_numeric(df['income'].str.replace(',', ''), errors='coerce')


In [116]:
df_users = df[personal_cols].drop_duplicates(subset='iid').copy()
df_users.info()
df_users.head(20)

<class 'pandas.core.frame.DataFrame'>
Index: 551 entries, 0 to 8356
Data columns (total 44 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   iid       551 non-null    int64  
 1   gender    551 non-null    int64  
 2   age       543 non-null    float64
 3   field     545 non-null    object 
 4   field_cd  544 non-null    float64
 5   undergra  315 non-null    object 
 6   race      545 non-null    float64
 7   imprace   544 non-null    float64
 8   imprelig  544 non-null    float64
 9   from      544 non-null    object 
 10  zipcode   475 non-null    object 
 11  income    281 non-null    float64
 12  goal      544 non-null    float64
 13  date      543 non-null    float64
 14  go_out    544 non-null    float64
 15  career_c  539 non-null    float64
 16  sports    544 non-null    float64
 17  tvsports  544 non-null    float64
 18  exercise  544 non-null    float64
 19  dining    544 non-null    float64
 20  museums   544 non-null    float64
 2

,iid,gender,age,field,field_cd,undergra,race,imprace,imprelig,from,zipcode,income,goal,date,go_out,career_c,sports,tvsports,exercise,dining,museums,art,hiking,gaming,clubbing,reading,tv,theater,movies,concerts,music,shopping,yoga,attr1_1,sinc1_1,intel1_1,fun1_1,amb1_1,shar1_1,attr3_1,sinc3_1,intel3_1,fun3_1,amb3_1
0,1,0,21.0,Law,1.0,NaN,4.0,2.0,4.0,Chicago,"60,521",69487.0,2.0,7.0,1.0,NaN,9.0,2.0,8.0,9.0,1.0,1.0,5.0,1.0,5.0,6.0,9.0,1.0,10.0,10.0,9.0,8.0,1.0,15.00,20.00,20.00,15.00,15.00,15.00,6.0,8.0,8.0,8.0,7.0
10,2,0,24.0,law,1.0,NaN,2.0,2.0,5.0,Alabama,"35,223",65929.0,1.0,5.0,1.0,NaN,3.0,2.0,7.0,10.0,8.0,6.0,3.0,5.0,8.0,10.0,1.0,9.0,8.0,7.0,8.0,3.0,1.0,45.00,5.00,25.00,20.00,0.00,5.00,7.0,5.0,8.0,10.0,3.0
20,3,0,25.0,Economics,2.0,NaN,2.0,8.0,4.0,Connecticut,"6,268",NaN,6.0,3.0,1.0,NaN,3.0,8.0,7.0,8.0,5.0,5.0,8.0,4.0,5.0,7.0,8.0,7.0,7.0,7.0,5.0,8.0,7.0,35.00,10.00,35.00,10.00,10.00,0.00,8.0,9.0,9.0,8.0,8.0
30,4,0,23.0,Law,1.0,NaN,2.0,1.0,1.0,Texas,"77,096",37754.0,1.0,5.0,1.0,1.0,1.0,1.0,6.0,7.0,6.0,7.0,7.0,5.0,7.0,7.0,7.0,9.0,7.0,8.0,7.0,1.0,8.0,20.00,20.00,20.00,20.00,10.00,10.00,7.0,8.0,7.0,9.0,8.0
40,5,0,21.0,Law,1.0,NaN,2.0,8.0,1.0,Bowdoin College,"94,022",86340.0,2.0,4.0,1.0,1.0,7.0,4.0,7.0,7.0,6.0,8.0,6.0,6.0,8.0,6.0,8.0,6.0,6.0,3.0,7.0,8.0,3.0,20.00,5.00,25.00,25.00,10.00,15.00,6.0,3.0,10.0,6.0,8.0
50,6,0,23.0,law,1.0,NaN,4.0,1.0,1.0,MD,"20,878",60304.0,1.0,3.0,1.0,1.0,10.0,8.0,9.0,7.0,8.0,7.0,9.0,2.0,6.0,9.0,2.0,5.0,6.0,6.0,4.0,1.0,1.0,10.00,25.00,20.00,25.00,5.00,15.00,5.0,7.0,9.0,8.0,5.0
60,7,0,22.0,Law,1.0,NaN,4.0,2.0,4.0,Southern California,"91,360",54620.0,1.0,5.0,1.0,1.0,5.0,3.0,4.0,10.0,10.0,10.0,2.0,3.0,8.0,8.0,8.0,10.0,10.0,10.0,10.0,10.0,10.0,15.00,15.00,25.00,20.00,15.00,10.00,6.0,6.0,7.0,5.0,7.0
70,8,0,25.0,Masters in Public Administration,13.0,NaN,2.0,1.0,1.0,"London, England",0,NaN,1.0,5.0,1.0,6.0,2.0,2.0,1.0,10.0,9.0,9.0,3.0,2.0,10.0,8.0,10.0,9.0,9.0,6.0,6.0,8.0,6.0,9.09,18.18,27.27,18.18,18.18,9.09,7.0,4.0,8.0,8.0,8.0
80,9,0,26.0,Masters in Public Administration,13.0,NaN,6.0,1.0,1.0,"Palm Springs, California",0,NaN,1.0,4.0,1.0,9.0,4.0,3.0,1.0,8.0,6.0,7.0,2.0,2.0,10.0,8.0,8.0,10.0,10.0,9.0,9.0,8.0,3.0,20.00,10.00,20.00,30.00,10.00,10.00,7.0,6.0,7.0,10.0,7.0
90,10,0,26.0,Masters of Social Work&Education,13.0,NaN,2.0,4.0,4.0,94115,"19,335",48652.0,2.0,6.0,1.0,9.0,9.0,9.0,9.0,7.0,6.0,6.0,7.0,1.0,7.0,8.0,5.0,6.0,7.0,7.0,8.0,7.0,7.0,15.00,15.00,15.00,40.00,10.00,5.00,6.0,8.0,6.0,10.0,9.0


In [115]:
df_interactions = df[interaction_cols].copy()
df_interactions['pid'] = df_interactions['pid'].astype(int)
df_interactions.info()
df_interactions.head(20)

<class 'pandas.core.frame.DataFrame'>
Index: 8368 entries, 0 to 8377
Data columns (total 23 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   iid       8368 non-null   int64  
 1   pid       8368 non-null   int64  
 2   int_corr  8210 non-null   float64
 3   samerace  8368 non-null   int64  
 4   dec_o     8368 non-null   int64  
 5   attr_o    8166 non-null   float64
 6   sinc_o    8091 non-null   float64
 7   intel_o   8072 non-null   float64
 8   fun_o     8018 non-null   float64
 9   amb_o     7656 non-null   float64
 10  shar_o    7302 non-null   float64
 11  like_o    8128 non-null   float64
 12  prob_o    8060 non-null   float64
 13  dec       8368 non-null   int64  
 14  attr      8166 non-null   float64
 15  sinc      8091 non-null   float64
 16  intel     8072 non-null   float64
 17  fun       8018 non-null   float64
 18  amb       7656 non-null   float64
 19  shar      7302 non-null   float64
 20  like      8128 non-null   float64
 

,iid,pid,int_corr,samerace,dec_o,attr_o,sinc_o,intel_o,fun_o,amb_o,shar_o,like_o,prob_o,dec,attr,sinc,intel,fun,amb,shar,like,prob,match
0,1,11,0.14,0,0,6.0,8.0,8.0,8.0,8.0,6.0,7.0,4.0,1,6.0,9.0,7.0,7.0,6.0,5.0,7.0,6.0,0
1,1,12,0.54,0,0,7.0,8.0,10.0,7.0,7.0,5.0,8.0,4.0,1,7.0,8.0,7.0,8.0,5.0,6.0,7.0,5.0,0
2,1,13,0.16,1,1,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,1,5.0,8.0,9.0,8.0,5.0,7.0,7.0,NaN,1
3,1,14,0.61,0,1,7.0,8.0,9.0,8.0,9.0,8.0,7.0,7.0,1,7.0,6.0,8.0,7.0,6.0,8.0,7.0,6.0,1
4,1,15,0.21,0,1,8.0,7.0,9.0,6.0,9.0,7.0,8.0,6.0,1,5.0,6.0,7.0,7.0,6.0,6.0,6.0,6.0,1
5,1,16,0.25,0,1,7.0,7.0,8.0,8.0,7.0,7.0,7.0,6.0,0,4.0,9.0,7.0,4.0,6.0,4.0,6.0,5.0,0
6,1,17,0.34,0,0,3.0,6.0,7.0,5.0,8.0,7.0,2.0,1.0,1,7.0,6.0,7.0,4.0,6.0,7.0,6.0,5.0,0
7,1,18,0.50,0,0,6.0,7.0,5.0,6.0,8.0,6.0,7.0,5.0,0,4.0,9.0,7.0,6.0,5.0,6.0,6.0,7.0,0
8,1,19,0.28,0,1,7.0,7.0,8.0,8.0,8.0,9.0,6.5,8.0,1,7.0,6.0,8.0,9.0,8.0,8.0,7.0,7.0,1
9,1,20,-0.36,0,0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,1,5.0,6.0,6.0,8.0,10.0,8.0,6.0,6.0,0


In [ ]:
data_numeric = data.select_dtypes(include=['number'])
print(f"Dropped columns: {data.columns.difference(data_numeric.columns).tolist()}")

In [ ]:
# Drop non-numeric columns
data_numeric = data.select_dtypes(include=['object'])
print(data_numeric.info())
print(data_numeric.isnull().mean())
# data_numeric.head(50)
# in tỷ lệ null




# EDA auto


In [ ]:
from ydata_profiling import ProfileReport

# Pass the DataFrame you want to perform EDA on to the ProfileReport function of Ydata-Profiling.
profile_df = ProfileReport(data,title="Businees Data Report", explorative=True)

# Executing this will automatically save a visualization report file in the current directory.
profile_df.to_notebook_iframe()